# A Custom Bayesian MMM with Directional Cross-Channel Interactions

Traditional Marketing Mix Models (MMM) assume that each paid channel contributes
to sales independently:

$$
y_t \;=\; \alpha
\;+\; \sum_i \beta_i \, m_{it}
\;+\; \text{controls}_t
\;+\; \text{seasonality}_t
\;+\; \text{holidays}_t
\;+\; \varepsilon_t,
$$

where $m_{it}$ is channel $i$'s adstocked / saturated spend at week $t$.

**This is a useful simplification, but an incomplete one.** In practice,
channels interact: an upper-funnel video burst makes paid-search clicks more
efficient, a brand campaign lifts the response to a CRM blast, and overlapping
campaigns can cannibalise one another. Traditional MMM effectively silos every
channel and routes these interaction effects into either the residual or an
inflated baseline.

---

### The directional-interaction model

We extend the standard formulation with a multiplicative, directional
cross-channel modifier:

$$
\boxed{\;
y^{\text{scaled}}_t \;=\; \alpha
\;+\; \sum_i \beta_i \, m_{it} \,
\exp\!\Big(\sum_{j \ne i} \gamma_{ij}\, m_{jt}\Big)
\;+\; \text{controls}_t
\;+\; \text{fourier}_t
\;+\; \text{holidays}_t
\;+\; \varepsilon_t
\;}
$$

Parameter intuition (all on the **mean-scaled target** scale):

- $\beta_i$ is the **silo / baseline effectiveness** of channel $i$ —
  the standalone contribution component attributable to channel $i$
  before any cross-channel amplification or suppression.

- $\gamma_{ij}$ is the **directional impact of channel $j$ on channel
  $i$'s effectiveness**.

- Because the interaction modifier is wrapped in $\exp(\cdot)$, the
  multiplicative media contribution always remains positive.

- Since transformed media inputs are max-scaled to approximately
  $(0,1)$, $\gamma_{ij}$ can be interpreted as the approximate
  **maximum log-multiplier** applied to channel $i$'s effectiveness when
  channel $j$ moves from zero to full normalized intensity.

  To see this, isolate a single interaction term:

  $$
  C_i
  =
  \beta_i m_i \exp(\gamma_{ij}m_j).
  $$

  Because:

  $$
  m_j \in [0,1],
  $$

  the interaction multiplier ranges between:

  $$
  \exp(\gamma_{ij}\cdot0)=1
  $$

  and:

  $$
  \exp(\gamma_{ij}\cdot1)=e^{\gamma_{ij}}.
  $$

  So:

  $$
  e^{\gamma_{ij}}
  $$

  is the maximum multiplicative effect channel $j$ can apply to channel
  $i$'s effectiveness over the observed media range.

  The corresponding proportional uplift is:

  $$
  e^{\gamma_{ij}} - 1.
  $$

  For example, if:

  $$
  \gamma_{12}=0.20,
  $$

  then:

  $$
  e^{0.20}\approx1.22,
  $$

  implying channel_2 can increase channel_1 effectiveness by up to
  approximately:

  $$
  1.22 - 1
  =
  22\%.
  $$

  Likewise, negative values naturally encode suppression or
  cannibalisation. For example:

  $$
  \gamma_{12}=-0.20
  $$

  implies channel_2 can reduce channel_1 effectiveness by up to:

  $$
  e^{-0.20}-1
  \approx
  -18\%.
  $$

- Crucially, $\gamma_{ij} \ne \gamma_{ji}$ in general. The model
  therefore captures **directional** synergy and cannibalisation,
  allowing one channel to amplify — or suppress — another asymmetrically.

## Why interactions are applied *after* adstock and saturation

A key modelling decision is **where** the cross-channel interaction layer should enter the MMM.

In this notebook, interactions are applied *after* each channel has already passed through:

1. max scaling,
2. geometric adstock,
3. nonlinear saturation.

That is:

$$
m_{i,t}
=
S_i(A_i(x_{i,t}))
$$

and the observed contribution becomes:

$$
C_{i,t}
=
\beta_i m_{i,t}
\exp\!\left(
\sum_{j \ne i}\gamma_{ij}m_{j,t}
\right).
$$

This means cross-channel interactions operate on **effective media pressure**, not raw spend.

---

### Why this matters

Suppose channel $j$ is already heavily saturated.

In a realistic media system, increasing spend further should not continue creating large interaction effects indefinitely. If additional spend in channel $j$ barely changes incremental exposure or awareness, then it should also barely change the effectiveness of other channels.

Applying interactions after saturation naturally captures this behaviour.

As channel $j$ saturates:

$$
m_{j,t}
=
S_j(A_j(x_{j,t}))
$$

begins to flatten, meaning additional spend produces only small increases in transformed media pressure.

As a result, the interaction multiplier:

$$
\exp(\gamma_{ij}m_{j,t})
$$

also stabilises.

This creates an intuitive and desirable property:

> interaction effects weaken naturally as channels saturate.

---

### Interpretation

Under this formulation:

$$
\beta_i m_{i,t}
$$

represents the standalone contribution from channel $i$, while:

$$
\exp\!\left(
\sum_{j \ne i}\gamma_{ij}m_{j,t}
\right)
$$

acts as an **effectiveness multiplier** driven by the surrounding media ecosystem.

Crucially, the interaction layer responds to:

- effective exposure,
- carryover-adjusted pressure,
- and saturated media intensity,

rather than raw spend itself.

This keeps the model behaviour realistic:

- low effective media pressure creates weak interactions,
- strong media presence amplifies interactions,
- heavily saturated channels stop generating large incremental synergy.

---

### Alternative formulation

An alternative approach would apply interactions *before* adstock and saturation:

$$
C_{i,t}
=
\beta_i
S_i\!\left(
A_i\!\left(
x_{i,t}
\exp\!\left(
\sum_{j \ne i}\gamma_{ij}x_{j,t}
\right)
\right)
\right).
$$

Under that formulation, interactions modify the **effective spend input** entering the response curve.

While mathematically valid, this approach is harder to interpret because the interaction effect becomes entangled with:

- carryover dynamics,
- nonlinear saturation,
- and the curvature of the response function itself.

By applying interactions after transformation, we preserve a much cleaner interpretation:

- $\beta_i$ remains the silo response,
- $\gamma_{ij}$ remains an effectiveness modifier,
- and contribution decomposition stays straightforward and interpretable.

## 1. Setup

Standard scientific Python stack plus `pymc`, `pymc-marketing`, and the
`Prior` helper. We follow current PyMC-Marketing conventions: `Prior` lives in
`pymc_extras.prior`.

In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pymc_marketing
import pymc_extras
import pytensor.tensor as pt
from pytensor.xtensor.type import as_xtensor
import seaborn as sns
import xarray as xr
from dateutil.easter import easter

# PyMC-Marketing primitives
from pymc_marketing.mmm import GeometricAdstock, TanhSaturation
from pymc_marketing.mmm.events import EventEffect, GaussianBasis
from pymc_marketing.mmm.fourier import YearlyFourier

from pymc_extras.prior import Prior

print(f"pymc version:           {pm.__version__}")
print(f"pymc-extras version:   {pymc_extras.__version__}")
print(f"pymc-marketing version: {pymc_marketing.__version__}")

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

SAMPLE_KWARGS = (
    dict(draws=1000, tune=1000, chains=4, target_accept=0.9, nuts_sampler = "numpyro")
)

## 2. Generate a synthetic dataset

We need data where the *truth* is known, including a sparse, directional
$\gamma$ matrix. We generate **3 years of weekly observations** for 4 media
channels and 2 control variables. The construction is calibrated so that a
correctly specified model recovers the truth comfortably (target
$R^2 \approx 0.85$, MAPE $\le 10\%$).

A few decisions worth flagging:

- **Holidays match the model's basis**: Christmas and Easter contributions are
  generated as Gaussian bumps centred on each event date, exactly mirroring the
  `GaussianBasis` used in inference. This is realistic and keeps the recovery
  clean.
- **Mean-scaled target**: we divide the simulated sales series by its mean so
  the model lives in $\sim 1$-centred space, which makes the intercept and
  $\beta$ priors trivially interpretable.
- **Max-scaled media** (each channel divided by its own max) so transformed
  media after adstock + saturation lives in $(0, 1)$ — and so $\gamma_{ij}$ is
  in the units described above.

In [ ]:
@dataclass
class SyntheticTruth:
    df: pd.DataFrame
    media_actual: pd.DataFrame
    media_scaled: pd.DataFrame
    controls_actual: pd.DataFrame
    controls_scaled: pd.DataFrame
    holiday_distance: pd.DataFrame
    fourier_features: pd.DataFrame
    y_actual: pd.Series
    y_scaled: pd.Series
    target_mean: float
    holiday_dates: dict[str, list[pd.Timestamp]]
    channel_names: list[str]
    control_names: list[str]
    holiday_names: list[str]


def _geometric_adstock_np(x: np.ndarray, alpha: float, l_max: int = 12) -> np.ndarray:
    # NumPy implementation matching pymc-marketing GeometricAdstock.
    n = len(x)
    weights = alpha ** np.arange(l_max + 1)
    out = np.zeros(n)
    for t in range(n):
        lags = x[max(0, t - l_max) : t + 1][::-1]
        out[t] = np.sum(weights[: len(lags)] * lags)
    return out


def _tanh_saturation_np(x: np.ndarray, b: float, c: float) -> np.ndarray:
    # NumPy implementation matching pymc-marketing TanhSaturation: b * tanh(x / (b * c)).
    return b * np.tanh(x / (b * c))


def _difference_in_days(model_dates: pd.DatetimeIndex, event_dates: pd.DatetimeIndex) -> np.ndarray:
    one_day = np.timedelta64(1, "D")
    return (model_dates.to_numpy()[:, None] - event_dates.to_numpy()) / one_day


def _nearest_event_distance(
    model_dates: pd.DatetimeIndex,
    holiday_dates: dict[str, list[pd.Timestamp]],
) -> np.ndarray:
    # For each date and holiday, return the signed distance (days) to the
    # nearest occurrence of that holiday. This is the X passed to EventEffect.
    out = np.zeros((len(model_dates), len(holiday_dates)))
    for j, (_, centers) in enumerate(holiday_dates.items()):
        diffs = _difference_in_days(model_dates, pd.DatetimeIndex(centers))
        nearest = np.argmin(np.abs(diffs), axis=1)
        out[:, j] = diffs[np.arange(len(model_dates)), nearest]
    return out


def _gaussian_bump(distance_days: np.ndarray, sigma_days: float) -> np.ndarray:
    # Same normalisation as GaussianBasis in pymc-marketing (a normal pdf).
    return (1.0 / (sigma_days * np.sqrt(2 * np.pi))) * np.exp(
        -0.5 * (distance_days / sigma_days) ** 2
    )


def generate_synthetic_data(
    n_weeks: int = 208,
    start_date: str = "2021-01-04",
    seed: int = RANDOM_SEED,
) -> SyntheticTruth:
    rng = np.random.default_rng(seed)

    dates = pd.date_range(start=start_date, periods=n_weeks, freq="W-MON")
    channel_names = [f"channel_{i+1}" for i in range(4)]
    control_names = ["control_1", "control_2"]
    holiday_names = ["christmas", "easter"]
    n_channels = len(channel_names)

    # --- media spend (actual units, GBP) ---------------------------------
    # Two ingredients are essential for gamma_ij to be statistically
    # identifiable from beta_i:
    #
    #   1. Channels must have INDEPENDENT temporal variation, so that
    #      m_i * m_j is not collinear with m_i alone.
    #   2. Each channel must have weeks where it is essentially DARK
    #      (m_j ~ 0). Without dark weeks, gamma_ij only ever scales an
    #      always-on multiplier and can be absorbed into beta_i.
    #
    # We achieve (1) with quarter-shifted seasonality plus heavy lognormal
    # noise and (2) with a per-channel binary "campaign active" mask:
    # ~30% of weeks for each channel are dark, drawn independently across
    # channels. This is realistic -- real campaigns turn on and off -- and
    # gives the model the contrast it needs to estimate gamma.
    base_levels = np.array([50_000.0, 35_000.0, 25_000.0, 15_000.0])
    phase_shift_weeks = np.array([0, 13, 26, 39])  # ~quarterly desync per channel
    media_actual = np.zeros((n_weeks, n_channels))
    week_idx = np.arange(n_weeks)
    campaign_active = rng.binomial(1, 0.7, size=(n_weeks, n_channels))
    for i, base in enumerate(base_levels):
        seasonal = 1.0 + 0.25 * np.sin(
            2 * np.pi * (week_idx + phase_shift_weeks[i]) / 52
        )
        noise = rng.lognormal(mean=0.0, sigma=0.30, size=n_weeks)
        burst_weeks = rng.choice(n_weeks, size=20, replace=False)
        burst = np.ones(n_weeks)
        burst[burst_weeks] *= rng.uniform(1.8, 2.6, size=burst_weeks.size)
        media_actual[:, i] = base * seasonal * noise * burst * campaign_active[:, i]
    media_actual = pd.DataFrame(media_actual, index=dates, columns=channel_names)

    # --- max-scale media -------------------------------------------------
    media_scaled = media_actual / media_actual.max()

    # --- "true" adstock + saturation transforms (deterministic) ----------
    # Adstock decays kept intentionally short (alpha <= 0.25) so that the
    # random dark weeks in the spend pattern remain "dark" after adstock:
    # a single dark week leaves m_i close to zero, which is what gives
    # gamma_ij its identifying contrast (weeks where m_j is dark let us
    # isolate channel_i's silo contribution beta_i * m_i).
    alpha_true = np.array([0.20, 0.10, 0.25, 0.05])
    sat_b_true = np.array([0.9, 1.0, 0.8, 1.1])
    sat_c_true = np.array([1.0, 0.9, 1.2, 1.0])
    transformed_media = np.zeros_like(media_scaled.values)
    for i in range(n_channels):
        adstocked = _geometric_adstock_np(media_scaled.values[:, i], alpha=alpha_true[i], l_max=12)
        transformed_media[:, i] = _tanh_saturation_np(adstocked, b=sat_b_true[i], c=sat_c_true[i])

    # --- directional gamma matrix ---------------------------------------
    # gamma[i, j] = effect of channel j on channel i.
    # The magnitudes below are deliberately chunky -- the per-week
    # interaction signal scales like beta_i * m_i * (exp(gamma_ij * m_j) - 1)
    # and beta_i is only ~0.1, so the smaller channels (3, 4) need bigger
    # gamma values to produce a recoverable cross-channel signal.
    beta_true = np.array([0.12, 0.10, 0.09, 0.08])
    gamma_true = np.zeros((n_channels, n_channels))
    # channel_2 strongly amplifies channel_1 (the headline directional effect)
    gamma_true[0, 1] = 0.50
    # the reverse direction is much smaller -- showcases directionality
    gamma_true[1, 0] = 0.20
    # channel_4 amplifies channel_3 strongly; channel_3 amplifies channel_4
    # weakly -- a second illustration of directionality.
    gamma_true[2, 3] = 0.80
    gamma_true[3, 2] = 0.25
    # channel_1 cannibalises channel_4 (the negative interaction)
    gamma_true[3, 0] = -0.60
    gamma_eff = gamma_true * (1 - np.eye(n_channels))

    # interaction multiplier and observed contribution
    log_mult = transformed_media @ gamma_eff.T  # (n_weeks, n_channels)
    interaction_multiplier = np.exp(log_mult)
    silo_contribution = transformed_media * beta_true  # broadcasting on channel
    observed_contribution = silo_contribution * interaction_multiplier
    media_signal = observed_contribution.sum(axis=1)

    # --- controls --------------------------------------------------------
    raw_controls = np.column_stack([
        1.0 + 0.25 * np.sin(2 * np.pi * week_idx / 52 + 0.4)
            + rng.normal(0.0, 0.15, size=n_weeks),  # control_1: macro index-ish
        1.0 + 0.10 * np.cos(2 * np.pi * week_idx / 26)
            + rng.normal(0.0, 0.10, size=n_weeks),  # control_2: shorter cycle
    ])
    controls_actual = pd.DataFrame(raw_controls, index=dates, columns=control_names)
    controls_scaled = controls_actual / controls_actual.mean()
    control_beta_true = np.array([0.06, -0.04])
    control_signal = (controls_scaled.values - 1.0) @ control_beta_true

    # --- Fourier seasonality --------------------------------------------
    dayofyear = dates.dayofyear.to_numpy()
    period = 365.25
    fourier_features = pd.DataFrame(
        {
            "sin_1": np.sin(2 * np.pi * 1 * dayofyear / period),
            "cos_1": np.cos(2 * np.pi * 1 * dayofyear / period),
            "sin_2": np.sin(2 * np.pi * 2 * dayofyear / period),
            "cos_2": np.cos(2 * np.pi * 2 * dayofyear / period),
        },
        index=dates,
    )
    fourier_beta_true = np.array([0.04, 0.02, -0.02, 0.015])
    # Note: YearlyFourier orders its features as ["sin_1", "sin_2", "cos_1", "cos_2"].
    # We carry that ordering explicitly.
    fourier_signal = fourier_features.values @ fourier_beta_true

    # --- holidays (Gaussian bumps that mirror GaussianBasis) ------------
    years = sorted(set(dates.year))
    holiday_dates = {
        "christmas": [pd.Timestamp(f"{y}-12-25") for y in years],
        "easter": [pd.Timestamp(easter(y)) for y in years],
    }
    holiday_distance_arr = _nearest_event_distance(dates, holiday_dates)
    holiday_amp_true = np.array([0.10, 0.05])
    holiday_sigma_true = np.array([7.0, 7.0])  # days
    holiday_signal = np.zeros(n_weeks)
    for j in range(len(holiday_names)):
        bump = _gaussian_bump(holiday_distance_arr[:, j], holiday_sigma_true[j])
        # GaussianBasis output peaks at 1/(sigma*sqrt(2*pi)); EventEffect = basis * effect_size.
        # So to land at amplitude `holiday_amp_true` at the peak, scale by sigma*sqrt(2*pi).
        scale = holiday_sigma_true[j] * np.sqrt(2 * np.pi)
        holiday_signal += holiday_amp_true[j] * bump * scale

    holiday_distance = pd.DataFrame(
        holiday_distance_arr, index=dates, columns=holiday_names
    )

    # --- assemble target -------------------------------------------------
    # We keep sigma_y at a level where the interaction signal is clearly
    # identifiable (per-week interaction magnitude ~ beta_i * m_i * gamma_ij
    # * m_j ~ 0.01 in mean-scaled units; we want sigma_y around the same
    # order of magnitude so gamma is recoverable in ~200 weeks of data).
    intercept_true = 0.8
    sigma_y_true = 0.02
    noise_eps = rng.normal(0.0, sigma_y_true, size=n_weeks)
    y_scaled_vec = (
        intercept_true
        + media_signal
        + control_signal
        + fourier_signal
        + holiday_signal
        + noise_eps
    )
    # Sales pivot: pick a target mean roughly 500k GBP so ROAS is meaningful
    target_mean = 500_000.0
    y_actual_vec = y_scaled_vec * target_mean
    y_scaled = pd.Series(y_scaled_vec, index=dates, name="y_scaled")
    y_actual = pd.Series(y_actual_vec, index=dates, name="y_actual")

    df = (
        media_actual.add_suffix("_spend")
        .join(controls_actual)
        .assign(y_actual=y_actual, y_scaled=y_scaled)
    )

    return SyntheticTruth(
        df=df,
        media_actual=media_actual,
        media_scaled=media_scaled,
        controls_actual=controls_actual,
        controls_scaled=controls_scaled,
        holiday_distance=holiday_distance,
        fourier_features=fourier_features,
        y_actual=y_actual,
        y_scaled=y_scaled,
        target_mean=target_mean,
        holiday_dates=holiday_dates,
        channel_names=channel_names,
        control_names=control_names,
        holiday_names=holiday_names,
    )


truth = generate_synthetic_data()
print(f"Weeks simulated: {len(truth.df)}")
print(f"Target mean (£): {truth.target_mean:,.0f}")
print(f"Channels: {truth.channel_names}")
print("Holiday dates kept:", {k: len(v) for k, v in truth.holiday_dates.items()})
truth.df.head()

## 3. Visualise the synthetic data

A quick four-pane sanity check: actual target, raw spend, scaled spend, and
the true $\gamma$ matrix.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(truth.y_actual.index, truth.y_actual.values, color="#1f77b4")
axes[0].set_title("Synthetic weekly sales (actual £)")
axes[0].set_ylabel("Sales (£)")

for col in truth.channel_names:
    axes[1].plot(truth.media_actual.index, truth.media_actual[col], label=col, alpha=0.8)
axes[1].set_title("Raw media spend by channel (actual £)")
axes[1].set_ylabel("Spend (£)")
axes[1].legend(loc="upper right", ncol=2, fontsize=9)

for col in truth.channel_names:
    axes[2].plot(truth.media_scaled.index, truth.media_scaled[col], label=col, alpha=0.8)
axes[2].set_title("Max-scaled media spend (model input space)")
axes[2].set_ylabel("Scaled spend")
axes[2].set_xlabel("Date")
axes[2].legend(loc="upper right", ncol=2, fontsize=9)

fig.tight_layout()
plt.show()

## 4. Parameter interpretation

Every parameter in the model lives on a known, **interpretable** scale because
we mean-scale the target and max-scale the media. The table below ties each
parameter to its real-world meaning.

| Parameter | Scale | Meaning | Interpretation example | Notes |
|---|---|---|---|---|
| $\alpha$ | mean-scaled target | baseline outcome when media/controls/seasonality/holidays are zero/reference | $\alpha = 0.8$ means baseline is 80% of average target | generated intercept is **0.8** |
| $\beta_i$ | mean-scaled target contribution | silo / base effectiveness of channel $i$ | $\beta_1 = 0.10$ means channel_1 can contribute up to $\sim 10\%$ of average target at max transformed media, before interactions | $\beta$ is **not** ROAS by itself |
| $\gamma_{ij}$ | log multiplier | directional effect of channel $j$ on channel $i$ | $\gamma_{12} = 0.50$ means channel_2 can increase channel_1's effectiveness by $e^{0.50} - 1 \approx 65\%$ at max transformed channel_2 | $\gamma_{ij}$ does **not** have to equal $\gamma_{ji}$ |
| $\beta^{\text{ctrl}}_k$ | mean-scaled target | effect of mean-scaled control $k$ | positive value means control increases target | controls are not media contribution |
| $\beta^{\text{hol}}_h$ | mean-scaled target | incremental holiday effect | Christmas may be positive or negative depending on the synthetic data | holiday effects are additive |
| $\sigma$ | mean-scaled target | residual noise | $\sigma = 0.02$ means residual sd is $\sim 2\%$ of average target | likelihood uncertainty |

## 5. Contribution definitions

With cross-channel interactions, "what did channel $i$ contribute?" admits
several legitimate answers. We track three, all stored as `pm.Deterministic`
nodes inside the model so they are saved to the trace and easy to query later.

### Silo contribution (scaled)

$$
C^{\text{silo}}_{it} \;=\; \beta_i \, m_{it}
$$

The contribution channel $i$ would receive under traditional, no-interaction
MMM logic — i.e. ignoring everything else that ran that week.

### Interaction multiplier

$$
M_{it} \;=\; \exp\!\Big(\sum_{j \ne i} \gamma_{ij}\, m_{jt}\Big)
$$

The realised effectiveness multiplier applied to channel $i$ because of the
*surrounding* media mix.

### Observed contribution (scaled)

$$
C^{\text{obs}}_{it} \;=\; C^{\text{silo}}_{it} \cdot M_{it}
$$

The realised contribution of channel $i$ under the actual observed media
ecosystem. **This is the reporting number** for business stakeholders.

### Interaction contribution (scaled)

$$
C^{\text{int}}_{it} \;=\; C^{\text{obs}}_{it} - C^{\text{silo}}_{it}
$$

Positive ⇒ channel $i$ was amplified by other channels. Negative ⇒ channel $i$
was suppressed / cannibalised.

### Actual-unit conversion

The target is mean-scaled before modelling, so to express any scaled
contribution in £ we simply multiply by `target_mean`:

$$
C^{\text{actual}}_{it} \;=\; C^{\text{scaled}}_{it} \cdot \bar y.
$$

| Contribution | Formula | Unit | Interpretation | Use case |
|---|---|---|---|---|
| **Silo** | $\beta_i\, m_{it}$ | actual target units (× `target_mean`) | Traditional MMM-style channel contribution | Compare to conventional MMM |
| **Observed** | $\beta_i\, m_{it}\, \exp(\sum_{j\ne i}\gamma_{ij} m_{jt})$ | actual target units | Realised contribution under actual media mix | Business reporting |
| **Interaction** | observed − silo | actual target units | Incremental uplift / suppression due to other channels | Synergy / cannibalisation analysis |
| **Total** | sum of all components | scaled (unless converted) | Fitted mean prediction / decomposition | Optimisation and AVM |

**Reporting rules of thumb**

- $\beta_i$ remains the *baseline* / silo effect — quoting $\beta$ alone is the
  cleanest comparison to a traditional MMM.
- $\gamma_{ij}$ changes *realised* effectiveness, not the baseline.
- **Observed contribution is the right number for "what did channel $i$
  drive?"** under the actual media mix.
- **Interaction contribution should not be treated as a fifth media channel.**
  It is an attribute of the existing channels, not an independently spendable
  unit.
- Because $\gamma$ is directional, "channel_2 → channel_1" and
  "channel_1 → channel_2" are reported separately.

## 6. Build the custom PyMC model

We work with a **raw `pm.Model`** rather than the `MMM` class so we can wire in
the directional cross-channel multiplier explicitly. The recipe:

1. Wrap inputs in `pm.Data` so we could later swap them for new spend plans.
2. Apply `GeometricAdstock(l_max=12)` and `TanhSaturation()` per channel,
   with **tight calibrated priors** on the adstock decay $\alpha$ and the
   saturation parameters $b, c$. In production this calibration would come
   from lift tests or geo experiments; here we use truth-adjacent values to
   reflect that real-world workflow. Leaving these parameters loose lets
   the saturation curve absorb cross-channel modulation and makes $\gamma$
   unidentifiable.
3. Build the full directional $\gamma$ matrix with `dims=("channel",
   "modifier_channel")`, then mask the diagonal so a channel cannot modify
   itself.
4. Wire the interaction multiplier $M_{it} = \exp(\sum_{j \ne i} \gamma_{ij}
   m_{jt})$ and the contribution stack — silo, observed, interaction, all in
   both scaled and actual units — as `pm.Deterministic` nodes.
5. Add Fourier seasonality via `YearlyFourier(n_order=2)` and holidays via
   `EventEffect(GaussianBasis(...), effect_size=Prior("Normal", 0, 0.05))`.
6. Likelihood: Normal on the mean-scaled target.

Prior choices are all on the mean-scaled-target scale. The structural
parameters $\alpha, \beta, \gamma$ are intentionally informative but proper;
the calibrated adstock / saturation parameters are pinned tightly to break
the identifiability tie between $\beta_i$, $\gamma_{ij}$, and the saturation
shape.

In [ ]:
n_dates = len(truth.df)
n_channels = len(truth.channel_names)

coords = {
    "date": truth.df.index.to_numpy(),
    "channel": truth.channel_names,
    "modifier_channel": truth.channel_names,
    "control": truth.control_names,
    "holiday": truth.holiday_names,
}

with pm.Model(coords=coords) as model:
    # ---------------------------------------------------------------- inputs
    media_data = pm.Data(
        "media_scaled", truth.media_scaled.values, dims=("date", "channel")
    )
    controls_data = pm.Data(
        "controls_scaled", truth.controls_scaled.values, dims=("date", "control")
    )
    holiday_distance_data = pm.Data(
        "holiday_distance", truth.holiday_distance.values, dims=("date", "holiday")
    )
    dayofyear = pm.Data(
        "dayofyear",
        truth.df.index.dayofyear.to_numpy().astype(float),
        dims="date",
    )
    target_mean = float(truth.target_mean)

    # ---------------------------------------------------------------- priors
    intercept = Prior("Normal", mu=0.8, sigma=0.15).create_variable("intercept")
    beta = Prior("HalfNormal", sigma=0.15, dims="channel").create_variable("beta")
    # gamma controls a log-multiplier. With media in (0, 1) a gamma of ~0.5
    # corresponds to a ~65% effectiveness lift at max -- a believable real
    # ceiling for synergy. We use sigma=0.40 here rather than a tighter
    # value because, given the noise level, a tighter prior dominates the
    # weak likelihood and shrinks genuine effects back toward zero. In
    # production, prefer a sparsity-inducing prior (horseshoe or
    # regularised Laplace) so unimportant entries get shrunk while truly
    # non-zero gamma_ij are free to escape.
    gamma = Prior(
        "Normal", mu=0.0, sigma=0.40, dims=("channel", "modifier_channel")
    ).create_variable("gamma")
    control_beta = Prior(
        "Normal", mu=0.0, sigma=0.10, dims="control"
    ).create_variable("control_beta")
    sigma = Prior("HalfNormal", sigma=0.03).create_variable("sigma")

    # mask the diagonal so a channel cannot modify itself
    gamma_eff = pm.Deterministic(
        "gamma_eff",
        gamma * (1.0 - pt.eye(n_channels)),
        dims=("channel", "modifier_channel"),
    )

    # ------------------------------------------------ adstock + saturation
    # In production we would calibrate the adstock decay and saturation
    # shape *outside* the structural model -- e.g. from lift tests or
    # historic geo experiments -- and then estimate beta / gamma on top.
    # We mirror that here with tight, well-centred TruncatedNormal priors:
    # the model is permitted to wiggle each parameter by ~one standard
    # error around the calibrated value, but the saturation curve cannot
    # contort itself to absorb the cross-channel multiplicative signal.
    # Without this tightening, beta, gamma, and the saturation b/c
    # parameters trade off against each other and gamma is unidentifiable.
    calibrated_alpha = np.array([0.20, 0.10, 0.25, 0.05])
    calibrated_b = np.array([0.9, 1.0, 0.8, 1.1])
    calibrated_c = np.array([1.0, 0.9, 1.2, 1.0])

    adstock = GeometricAdstock(
        l_max=12,
        priors={
            "alpha": Prior(
                "TruncatedNormal",
                mu=calibrated_alpha,
                sigma=0.05,
                lower=0.0,
                upper=1.0,
                dims="channel",
            )
        },
    )
    saturation = TanhSaturation(
        priors={
            "b": Prior(
                "TruncatedNormal",
                mu=calibrated_b,
                sigma=0.05,
                lower=0.0,
                dims="channel",
            ),
            "c": Prior(
                "TruncatedNormal",
                mu=calibrated_c,
                sigma=0.05,
                lower=0.0,
                dims="channel",
            ),
        }
    )
    media_x = as_xtensor(media_data, dims=("date", "channel"))
    adstocked_media_x = adstock.apply(media_x, core_dim="date")
    transformed_media_x = saturation.apply(adstocked_media_x, core_dim="date")
    transformed_media = pm.Deterministic(
        "transformed_media",
        transformed_media_x.transpose("date", "channel").values,
        dims=("date", "channel"),
    )

    # -------------------------------------------- contribution decomposition
    silo_contribution_scaled = pm.Deterministic(
        "silo_contribution_scaled",
        transformed_media * beta,  # broadcasts beta over the date dim
        dims=("date", "channel"),
    )

    log_multiplier = pt.dot(transformed_media, gamma_eff.T)
    interaction_multiplier = pm.Deterministic(
        "interaction_multiplier",
        pt.exp(log_multiplier),
        dims=("date", "channel"),
    )

    observed_contribution_scaled = pm.Deterministic(
        "observed_contribution_scaled",
        silo_contribution_scaled * interaction_multiplier,
        dims=("date", "channel"),
    )
    interaction_contribution_scaled = pm.Deterministic(
        "interaction_contribution_scaled",
        observed_contribution_scaled - silo_contribution_scaled,
        dims=("date", "channel"),
    )

    # actual-unit versions
    pm.Deterministic(
        "silo_contribution_actual",
        silo_contribution_scaled * target_mean,
        dims=("date", "channel"),
    )
    pm.Deterministic(
        "observed_contribution_actual",
        observed_contribution_scaled * target_mean,
        dims=("date", "channel"),
    )
    pm.Deterministic(
        "interaction_contribution_actual",
        interaction_contribution_scaled * target_mean,
        dims=("date", "channel"),
    )

    # ------------------------------------------------------ controls / fourier
    control_contribution_scaled = pm.Deterministic(
        "control_contribution_scaled",
        pt.dot(controls_data - 1.0, control_beta),
        dims="date",
    )

    fourier = YearlyFourier(
        n_order=2,
        prior=Prior("Normal", mu=0.0, sigma=0.05, dims="fourier"),
    )
    dayofyear_x = as_xtensor(dayofyear, dims=("date",))
    fourier_contribution_scaled = pm.Deterministic(
        "fourier_contribution_scaled",
        fourier.apply(dayofperiod=dayofyear_x).values,
        dims="date",
    )

    # ----------------------------------------------------- holidays / events
    gaussian_basis = GaussianBasis(
        priors={"sigma": Prior("Gamma", mu=7.0, sigma=1.0, dims="holiday")}
    )
    holiday_effect_size = Prior(
        "Normal", mu=0.0, sigma=0.05, dims="holiday"
    )
    event_effect = EventEffect(
        basis=gaussian_basis, effect_size=holiday_effect_size, dims=("holiday",)
    )
    holiday_distance_x = as_xtensor(
        holiday_distance_data, dims=("date", "holiday")
    )
    holiday_contribution_scaled = pm.Deterministic(
        "holiday_contribution_scaled",
        event_effect.apply(holiday_distance_x, name="holiday")
        .transpose("date", "holiday")
        .values,
        dims=("date", "holiday"),
    )

    # ------------------------------------------------------------------ mu
    contribution = pm.Deterministic(
        "contribution",
        intercept
        + observed_contribution_scaled.sum(axis=-1)
        + control_contribution_scaled
        + fourier_contribution_scaled
        + holiday_contribution_scaled.sum(axis=-1),
        dims="date",
    )

    # convenience: mean prediction in actual units (used by the AVM section)
    pm.Deterministic("y_mean", contribution * target_mean, dims="date")

    # likelihood
    pm.Normal(
        "y_obs",
        mu=contribution,
        sigma=sigma,
        observed=truth.y_scaled.values,
        dims="date",
    )

print("Model RVs:", [v.name for v in model.free_RVs])
print("Deterministics:", [v.name for v in model.deterministics])

## 7. Sample the posterior

We sample with NUTS, 4 chains, and `target_accept=0.9` because the multiplicative
interaction term can produce sharp posterior geometry. Posterior- and prior-
predictive draws are pulled in the same `with model:` block so we have
everything we need for downstream cells.

In [ ]:
with model:
    idata = pm.sample(
        random_seed=RANDOM_SEED,
        return_inferencedata=True,
        progressbar=False,
        **SAMPLE_KWARGS,
    )
    idata = pm.sample_posterior_predictive(
        idata,
        var_names=[
            "y_obs",
            "y_mean",
            "contribution",
            "silo_contribution_scaled",
            "observed_contribution_scaled",
            "interaction_contribution_scaled",
            "silo_contribution_actual",
            "observed_contribution_actual",
            "interaction_contribution_actual",
            "control_contribution_scaled",
            "fourier_contribution_scaled",
            "holiday_contribution_scaled",
            "transformed_media",
            "gamma_eff",
            "interaction_multiplier",
        ],
        random_seed=RANDOM_SEED,
        extend_inferencedata=True,
        progressbar=True,
    )
    prior_predictive = pm.sample_prior_predictive(
        samples=500, random_seed=RANDOM_SEED
    )

az.summary(idata, var_names=["intercept", "sigma"], round_to=3)

In [ ]:
free_rv_names = [v.name for v in model.free_RVs]
az.summary(idata, var_names=free_rv_names, round_to=3)["r_hat"].max()

## 8. Model graph

A visual sanity-check that the directional-gamma wiring is right. The graph
shows every random variable and deterministic, including the masked
`gamma_eff` flowing into the interaction multiplier.

In [ ]:
try:
    graph = pm.model_to_graphviz(model)
    display(graph)
except Exception as exc:  # pragma: no cover -- graphviz not always installed
    print("Could not render the model graph; install graphviz to see it.")
    print(f"Error: {exc}")

> **Attribution-stealing warning.** $\gamma$ being recovered near zero is
> *not* the same as $\gamma_{ij}$ being absent in the data. The classic
> failure mode is that the silo coefficient $\beta_i$ silently *inflates*
> to absorb the multiplicative interaction signal — you'll see this as
> $\beta_i$ posterior means well above their truth (or external) values,
> combined with strongly negative $(\beta_i, \gamma_{ij})$ posterior
> correlations. Sanity-check by:
>
> - re-running with a tighter prior on $\gamma$ and comparing $\beta$ and
>   silo contributions,
> - holding out a period and comparing silo vs observed contributions, and
> - sanity-checking interaction signs against domain knowledge.

> **A note on $\gamma$ priors and signal-to-noise.** The per-week
> interaction signal for a single $\gamma_{ij}$ is roughly
> $\beta_i \cdot m_{it} \cdot (e^{\gamma_{ij} m_{jt}} - 1)$. With media in
> $(0, 1)$ and $\beta \approx 0.1$, a $\gamma$ of $0.1$ produces a signal
> of only $\sim 0.001$ per week — well below typical noise floors. Two
> design choices in this notebook make $\gamma$ identifiable at all:
>
> 1. **Decorrelated channel activity** — each channel has random dark
>    weeks combined with short adstock decay, so $m_j$ actually reaches
>    zero in some weeks. Those dark weeks let the model isolate
>    $\beta_i \cdot m_i$ and pin down $\beta_i$ before it has to fit
>    $\gamma_{ij}$.
> 2. **Calibrated adstock and saturation** — pinning $\alpha$ and the
>    Tanh $b, c$ parameters tightly stops the saturation curve from
>    absorbing cross-channel modulation.
>
> The $\gamma$ prior here is $\mathcal{N}(0, 0.40)$, which is loose
> enough that the data can find the moderate-to-large interactions,
> but tight enough to regularise the (many) noise entries. **In
> production, with real and noisier data, use a sparsity-inducing
> prior** (horseshoe or regularised Laplace) which gives shrinkage
> only to the unimportant entries while letting the genuinely non-zero
> $\gamma_{ij}$ escape.

## 10. Actual vs Modelled

We use the posterior **mean of `y_mean`** (= `contribution * target_mean`) as
the point prediction and report two flavours of fit quality:

- **MAPE** — mean absolute percentage error, the relevant headline metric for
  business stakeholders because it is in % of actual sales.
- **R²** — fraction of explained variance.

Good AVM fit does not prove causal validity — for an interaction MMM you must
*also* inspect $\beta$ / $\gamma$ posterior behaviour to rule out attribution
stealing.

In [ ]:
y_actual = truth.y_actual.values
y_pred_actual = idata.posterior["y_mean"].mean(("chain", "draw")).values

mape = np.mean(np.abs((y_actual - y_pred_actual) / y_actual)) * 100
r2 = 1 - np.sum((y_actual - y_pred_actual) ** 2) / np.sum(
    (y_actual - y_actual.mean()) ** 2
)

print(f"MAPE = {mape:.2f}%")
print(f"R^2  = {r2:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(truth.df.index, y_actual, label="actual", color="#1f77b4", linewidth=1.6)
ax.plot(truth.df.index, y_pred_actual, label="modelled (posterior mean)",
        color="#d62728", linewidth=1.6, linestyle="--")
ax.set_title("Actual vs Modelled Target")
ax.set_xlabel("Date")
ax.set_ylabel("Sales (£)")
ax.legend(loc="upper left")
ax.text(
    0.99, 0.05,
    f"MAPE = {mape:.2f}%\n$R^2$ = {r2:.3f}",
    transform=ax.transAxes,
    ha="right", va="bottom",
    fontsize=11,
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.85, edgecolor="grey"),
)
fig.tight_layout()
plt.show()

**Reading the AVM diagnostics**

- A low MAPE (single-digit %) and an $R^2$ above the mid-0.7s indicate a
  good *absolute* fit relative to the actual target scale.
- Good fit alone does **not** prove causal validity. For an interaction MMM
  you must additionally check that:
  - $\beta_i$ did not silently *inflate* (or, in some cases, collapse)
    to absorb the multiplicative interaction signal — that's the
    attribution-stealing failure mode flagged in section 9,
  - $\gamma$ posteriors look reasonable and not pathologically wide,
  - silo contributions remain stable when priors on $\gamma$ are
    tightened or loosened.

## 11. Contribution reporting

One bar per channel: the lower segment is the **silo** contribution
\(\beta_i\, m_{i,t}\,\bar y\) summed over the window, the upper segment is
the **interaction** uplift (observed minus silo), so the total height is
the channel's observed contribution to sales over the window. Both
quantities are read from per-draw `pm.Deterministic` nodes and reduced to
a point estimate by averaging over the full posterior.

In [ ]:
# Per-channel posterior-mean contributions in £, reduced from full posterior.
silo_per_ch = (
    idata.posterior["silo_contribution_actual"].mean(("chain", "draw")).values
)  # (date, channel)
observed_per_ch = (
    idata.posterior["observed_contribution_actual"].mean(("chain", "draw")).values
)
silo_total = silo_per_ch.sum(axis=0)
observed_total = observed_per_ch.sum(axis=0)
interaction_total = observed_total - silo_total

silo_m = silo_total / 1e6
inter_m = interaction_total / 1e6
obs_m = observed_total / 1e6

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(truth.channel_names))
ax.bar(x_pos, silo_m, color="#1f77b4", label="silo")
ax.bar(
    x_pos, inter_m, bottom=silo_m,
    color="#ff7f0e", label="interaction (observed - silo)",
)
for k, total in enumerate(obs_m):
    ax.text(x_pos[k], total, f" £{total:,.2f}M", ha="center", va="bottom", fontsize=9)
ax.set_xticks(x_pos)
ax.set_xticklabels(truth.channel_names)
ax.set_ylabel("Total contribution over the window (£ millions)")
ax.set_title("Per-channel contribution: silo + interaction = observed")
ax.legend(loc="upper left")
ax.margins(y=0.12)
fig.tight_layout()
plt.show()

## 12. ROAS table

Two ROAS numbers per channel:

- **Silo ROAS** — what the channel would have returned in isolation.
- **Observed ROAS** — what it actually returned under the realised media mix.

The gap is the interaction effect, expressed in £-of-revenue-per-£-of-spend.

In [ ]:
total_spend = truth.media_actual.sum(axis=0).values
silo_actual_total = silo_per_ch.sum(axis=0)
observed_actual_total = observed_per_ch.sum(axis=0)
interaction_actual_total = observed_actual_total - silo_actual_total

roas_df = pd.DataFrame(
    {
        "total_spend_£": total_spend,
        "silo_contribution_£": silo_actual_total,
        "observed_contribution_£": observed_actual_total,
        "interaction_contribution_£": interaction_actual_total,
        "silo_ROAS": silo_actual_total / total_spend,
        "observed_ROAS": observed_actual_total / total_spend,
    },
    index=truth.channel_names,
)

roas_df.style.format({
    "total_spend_£": "£{:,.0f}",
    "silo_contribution_£": "£{:,.0f}",
    "observed_contribution_£": "£{:,.0f}",
    "interaction_contribution_£": "£{:,.0f}",
    "silo_ROAS": "{:.2f}",
    "observed_ROAS": "{:.2f}",
})

## 13. Channel 1 contribution vs spend

A planner-friendly view: **how much weekly sales does each £ of channel_1
spend drive**, and how does that response curve shift when channel_2's
weekly spend changes?

Move the slider to set channel_2's weekly spend; the red curve is
channel_1's observed weekly contribution to sales as a function of
channel_1 spend at the chosen channel_2 level. The grey dashed curve is
the **silo** response — the same response with the cross-channel
multiplier turned off (i.e. \(\exp(\cdot)=1\)) — fixed in place as a
reference, so the gap between the two curves is the interaction uplift
(or suppression) attributable to channel_2 at the slider's level.

The plotted curve is

$$
C^{\text{obs}}_1(x_1, x_2)
\;=\;
\beta_1\, S_1\!\big(A_1(x_1)\big)\,
\exp\!\big(
\gamma_{12}\, S_2(A_2(x_2))
+
\gamma_{13}\,\bar m_3
+
\gamma_{14}\,\bar m_4
\big)\,
\cdot \bar y,
$$

where:

- \(A_i(\cdot)\) is the channel-\(i\) geometric adstock,
- \(S_i(\cdot)\) is the channel-\(i\) Tanh saturation,
- \(\bar y\) is the mean of the original target series used to convert
  back into actual units.

For geometric adstock:

$$
A_i(x_t)
=
\sum_{l=0}^{L}
\alpha_i^l x_{t-l},
$$

with carryover truncated at the model's chosen `l_max`.

For Tanh saturation:

$$
S_i(z)
=
b_i \tanh\!\left(
\frac{z}{b_i c_i}
\right),
$$

which creates diminishing marginal returns at high spend levels.

We evaluate the curve using the **posterior mean** of:

$$
\alpha_i,\;
b_i,\;
c_i,\;
\beta_i,\;
\gamma_{ij},
$$

which keeps the slider responsive by avoiding posterior resampling during
interaction.

---

## Why there are multiple possible response curves

In a traditional MMM, each channel has a single response curve:

$$
C_i(x_i)
=
\beta_i\, S_i(A_i(x_i)).
$$

The contribution from channel \(i\) depends only on its own spend.

In the interaction MMM, however, the response curve for channel \(i\)
depends on the surrounding media ecosystem:

$$
C_i(x_i)
=
\beta_i\,S_i(A_i(x_i))
\exp\!\left(
\sum_{j \ne i}
\gamma_{ij}m_j
\right).
$$

That means the realised shape of channel \(i\)'s response curve changes
depending on which other channels are active.

For a focal channel \(i\), there are:

$$
n-1
$$

possible modifier channels.

Each modifier channel has two possible states:

- included in the interaction environment,
- excluded from the interaction environment.

So the number of possible interaction environments is:

$$
2^{n-1}.
$$

Equivalently:

$$
\sum_{k=0}^{n-1}
\binom{n-1}{k}
=
2^{n-1},
$$

which follows directly from the binomial theorem.

Recall the binomial theorem:

$$
(a+b)^r
=
\sum_{k=0}^{r}
\binom{r}{k}
a^{r-k}b^k.
$$

Set:

$$
a=1,
\qquad
b=1,
\qquad
r=n-1.
$$

Then:

$$
(1+1)^{n-1}
=
\sum_{k=0}^{n-1}
\binom{n-1}{k}
1^{n-1-k}1^k.
$$

Since:

$$
1^{n-1-k}1^k = 1,
$$

this simplifies to:

$$
2^{n-1}
=
\sum_{k=0}^{n-1}
\binom{n-1}{k}.
$$

So for a focal channel in an \(n\)-channel system, there are:

$$
\boxed{
2^{n-1}
}
$$

possible realised response curves.

---

## Example: channel_1 with four total channels

Suppose the full media mix contains:

$$
\{1,2,3,4\}.
$$

If we focus on the response curve for channel_1, then the possible
modifier channels are:

$$
\{2,3,4\}.
$$

So there are \(3\) possible modifier channels.

We can count the possible response curves by asking how many ways there
are to choose:

- no modifiers,
- one modifier,
- two modifiers,
- all three modifiers.

This gives:

$$
\binom{3}{0}
+
\binom{3}{1}
+
\binom{3}{2}
+
\binom{3}{3}.
$$

Expanding each term:

### No modifiers

$$
\binom{3}{0} = 1
$$

This is the **silo curve**:

$$
\varnothing
$$

No other channels are active in the interaction multiplier.

---

### One-to-one interaction curves

$$
\binom{3}{1} = 3
$$

These are the three one-to-one interaction environments:

$$
\{2\},\quad \{3\},\quad \{4\}.
$$

---

### One-to-two interaction curves

$$
\binom{3}{2} = 3
$$

These are the three one-to-two interaction environments:

$$
\{2,3\},\quad \{2,4\},\quad \{3,4\}.
$$

---

### Fully observed interaction curve

$$
\binom{3}{3} = 1
$$

This is the fully observed media ecosystem:

$$
\{2,3,4\}.
$$

---

Putting everything together:

$$
\binom{3}{0}
+
\binom{3}{1}
+
\binom{3}{2}
+
\binom{3}{3}
=
1 + 3 + 3 + 1
=
8.
$$

So channel_1 has:

$$
2^3 = 8
$$

possible realised response curves.

The eight possible environments are:

| Curve | Active modifiers |
|---|---|
| Silo curve | \(\varnothing\) |
| Channel_1 with channel_2 | \(\{2\}\) |
| Channel_1 with channel_3 | \(\{3\}\) |
| Channel_1 with channel_4 | \(\{4\}\) |
| Channel_1 with channels_2,3 | \(\{2,3\}\) |
| Channel_1 with channels_2,4 | \(\{2,4\}\) |
| Channel_1 with channels_3,4 | \(\{3,4\}\) |
| Channel_1 with channels_2,3,4 | \(\{2,3,4\}\) |

Each interaction environment produces a different realised response curve
because the exponential interaction multiplier changes the effective
productivity of channel_1.

More generally, for \(n\) channels:

$$
\boxed{
\text{Number of possible response curves per channel}
=
2^{n-1}
}
$$

and across all channels:

$$
\boxed{
n\,2^{n-1}
}
$$

possible realised response curves exist in total.

In practice, however, visualising all possible curves quickly becomes
unwieldy as \(n\) grows. Interactive sliders therefore provide a more
practical way to explore how one channel's response changes under
different media environments.

### Reading the two plots below

The widget that follows lets you pick any **focal channel** and then renders two response-curve views side by side:

- **Plot 1 - static atlas.** Every one of the \(2^{n-1}\) possible response curves for the chosen focal channel, one per modifier subset \(S \subseteq \{j \ne i\}\). The two extremes are drawn in **bold**: the dashed grey curve is the **silo** response (no modifiers active, \(\exp(\cdot) = 1\)), and the solid black curve is the fully **observed** response (all modifiers active). The intermediate \(2^{n-1} - 2\) curves are thin coloured lines. Each active modifier is pinned at its *historical mean weekly spend*; inactive modifiers contribute \(0\) to the log multiplier.

- **Plot 2 - one curve, your dials.** Always shows the silo curve plus a single non-silo curve picked from a dropdown. A spend dial appears for each modifier in the selected subset so you can sweep its weekly £ live - as you drag every dial down to £0, the red curve collapses onto silo, which is the visual sanity check that the interaction multiplier really does reduce to \(1\) when modifier media goes to zero.

Both panels overlay **historical points** for the focal channel - grey dots are weekly (spend, silo contribution) and black dots are weekly (spend, observed contribution), pulled directly from `idata.posterior`. The x-axis extends to \(5 \times\) the historical maximum spend so you can see each curve's diminishing-returns tail well past the realised range.

**Why the scatter doesn't sit cleanly on the curves.** The curves are *steady-state* response curves: they answer "if this channel spent \(x\) every week forever, what would the contribution be?" Geometric adstock at constant \(x\) collapses to \(x \sum_{\ell=0}^{L-1} \alpha^\ell\), so the curve effectively maps an adstock *output* level (re-expressed in spend units via the steady-state inverse) onto contribution. The scatter plots each historical week at its **raw weekly spend**, but the model actually saw that week's *adstocked* spend - a weighted sum of the previous \(\sim 12\) weeks - which can be much higher or lower than this week's raw spend depending on recent activity. If you replaced the x-coordinate of each historical point with the steady-state-equivalent spend \(x_{\text{eff}}[t] = (1-\alpha)\,\text{adstock}_{\text{actual}}[t]\), the silo points would land on the silo curve. We deliberately keep raw spend on the x-axis here because that's the lever a planner controls; the gap between scatter and curve is then a visual reminder that adstock is doing real work. Both the curves and the scatter are computed from the full posterior and then averaged - the per-draw response is evaluated on the grid and reduced to a point estimate at the end, so the rendered curve is \(\mathbb E_\theta[f(\theta)]\) rather than the Jensen-biased \(f(\mathbb E_\theta[\theta])\).

In [ ]:
import itertools
import ipywidgets as widgets
from IPython.display import display
from matplotlib.ticker import FuncFormatter

post = idata.posterior.stack(sample=("chain", "draw"))

# Per-draw posterior arrays, shape (D, ...) where D = chain * draw. Curves are
# computed for every draw and then averaged at the end, so the rendered curve
# equals E_theta[f(theta)] rather than the Jensen-biased f(E_theta[theta]).
beta_arr = post["beta"].transpose("sample", ...).values            # (D, C)
gamma_eff_arr = post["gamma_eff"].transpose("sample", ...).values  # (D, C, C); diagonal already 0
alpha_arr = post["adstock_alpha"].transpose("sample", ...).values  # (D, C)
b_arr = post["saturation_b"].transpose("sample", ...).values       # (D, C)
c_arr = post["saturation_c"].transpose("sample", ...).values       # (D, C)
silo_hist = (
    idata.posterior["silo_contribution_actual"].mean(("chain", "draw")).values
)
obs_hist = (
    idata.posterior["observed_contribution_actual"].mean(("chain", "draw")).values
)

D = beta_arr.shape[0]
channel_names = list(truth.channel_names)
n_ch = len(channel_names)
hist_actual = truth.media_actual.values
hist_max = hist_actual.max(axis=0)
hist_mean = hist_actual.mean(axis=0)
target_mean_v = float(truth.target_mean)
L_MAX = 12
adstock_ss_arr = sum(alpha_arr ** l for l in range(L_MAX + 1))  # (D, C)


def transformed_media_at(k: int, spend):
    # Returns shape (D, G) if spend is array-like, else (D,) for scalar spend.
    x = np.asarray(spend, dtype=float) / hist_max[k]
    bc = (b_arr[:, k] * c_arr[:, k])  # (D,)
    if x.ndim == 0:
        u = x * adstock_ss_arr[:, k] / bc
        return b_arr[:, k] * np.tanh(u)
    u = x[None, :] * (adstock_ss_arr[:, k] / bc)[:, None]
    return b_arr[:, k][:, None] * np.tanh(u)


def curve_for(focal: int, active, modifier_spends: dict):
    x_grid = np.linspace(0.0, 5.0 * hist_max[focal], 200)
    m_focal = transformed_media_at(focal, x_grid)   # (D, G)
    log_mult = np.zeros(D)
    for j in active:
        spend_j = modifier_spends.get(j, hist_mean[j])
        log_mult = log_mult + gamma_eff_arr[:, focal, j] * transformed_media_at(j, spend_j)
    curve = beta_arr[:, focal][:, None] * m_focal * np.exp(log_mult)[:, None]  # (D, G)
    return x_grid, curve.mean(axis=0) * target_mean_v


def _gbp(value, _pos):
    if value >= 1_000:
        return f"\u00a3{value / 1_000:,.0f}k"
    return f"\u00a3{value:,.0f}"


def _modifier_indices(focal: int):
    return tuple(j for j in range(n_ch) if j != focal)


def _all_subsets(focal: int):
    mods = _modifier_indices(focal)
    return [
        tuple(s)
        for r in range(len(mods) + 1)
        for s in itertools.combinations(mods, r)
    ]


def _subset_label(subset):
    if not subset:
        return "silo"
    return "{" + ", ".join(channel_names[j] for j in subset) + "}"


channel_dd = widgets.Dropdown(
    options=channel_names,
    value=channel_names[0],
    description="focal channel",
    style={"description_width": "initial"},
)

subset_dd = widgets.Dropdown(
    description="curve in plot 2",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="50%"),
)

modifier_sliders = {
    j: widgets.FloatSlider(
        value=hist_mean[j],
        min=0.0,
        max=5.0 * hist_max[j],
        step=hist_max[j] / 40 if hist_max[j] > 0 else 1.0,
        description=f"{channel_names[j]} \u00a3/wk",
        continuous_update=False,
        readout_format=",.0f",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="48%"),
    )
    for j in range(n_ch)
}

slider_box = widgets.HBox(
    list(modifier_sliders.values()),
    layout=widgets.Layout(flex_flow="row wrap"),
)

show_silo_cb = widgets.Checkbox(
    value=False,
    description="show historical silo observations",
    indent=False,
)
show_obs_cb = widgets.Checkbox(
    value=False,
    description="show historical observed observations",
    indent=False,
)
toggle_box = widgets.HBox([show_silo_cb, show_obs_cb])


def _refresh_subset_options(*_):
    focal = channel_names.index(channel_dd.value)
    non_silo = [s for s in _all_subsets(focal) if len(s) > 0]
    subset_dd.options = [(_subset_label(s), s) for s in non_silo]
    subset_dd.value = non_silo[-1]  # default to fully-observed subset


def _sync_slider_visibility(*_):
    focal = channel_names.index(channel_dd.value)
    active = set(subset_dd.value or ())
    for j, slider in modifier_sliders.items():
        if j == focal:
            slider.layout.display = "none"
        else:
            slider.layout.display = ""
            slider.disabled = j not in active


channel_dd.observe(_refresh_subset_options, names="value")
channel_dd.observe(_sync_slider_visibility, names="value")
subset_dd.observe(_sync_slider_visibility, names="value")
_refresh_subset_options()
_sync_slider_visibility()


def _render(focal_name, subset, show_silo, show_obs, **modifier_kwargs):
    focal = channel_names.index(focal_name)
    if subset is None:
        return
    modifier_spends = {
        j: modifier_kwargs[f"spend_{j}"]
        for j in subset
        if f"spend_{j}" in modifier_kwargs
    }

    subsets = _all_subsets(focal)
    n_mods = n_ch - 1

    peak_candidates = []
    static_curves = []
    for S in subsets:
        x_grid, y_grid = curve_for(focal, S, modifier_spends={})
        static_curves.append((S, x_grid, y_grid))
        peak_candidates.append(float(y_grid.max()))

    _, y_selected = curve_for(focal, subset, modifier_spends)
    peak_candidates.append(float(y_selected.max()))
    if show_obs:
        peak_candidates.append(float(obs_hist[:, focal].max()))
    if show_silo:
        peak_candidates.append(float(silo_hist[:, focal].max()))
    y_top = 1.08 * max(peak_candidates) if peak_candidates else 1.0
    x_top = 5.0 * hist_max[focal]

    fig, (ax_static, ax_dyn) = plt.subplots(2, 1, figsize=(13, 11))

    fully_observed_S = _modifier_indices(focal)
    for S, x_grid, y_grid in static_curves:
        if len(S) == 0:
            ax_static.plot(
                x_grid, y_grid,
                color="#555", linewidth=3.0, linestyle="--",
                label=_subset_label(S), zorder=4,
            )
        elif S == fully_observed_S:
            ax_static.plot(
                x_grid, y_grid,
                color="black", linewidth=3.0,
                label=f"observed {_subset_label(S)}", zorder=5,
            )
        else:
            ax_static.plot(
                x_grid, y_grid,
                linewidth=1.4,
                label=_subset_label(S), zorder=2,
            )

    if show_silo:
        ax_static.scatter(
            hist_actual[:, focal], silo_hist[:, focal],
            color="#888", s=14, alpha=0.7, zorder=6,
            label="historical (silo)",
        )
    if show_obs:
        ax_static.scatter(
            hist_actual[:, focal], obs_hist[:, focal],
            color="black", s=22, alpha=0.85, zorder=7,
            label="historical (observed)",
        )

    ax_static.set_xlabel(f"{focal_name} weekly spend (\u00a3)")
    ax_static.set_ylabel(f"{focal_name} weekly contribution to sales (\u00a3)")
    ax_static.xaxis.set_major_formatter(FuncFormatter(_gbp))
    ax_static.yaxis.set_major_formatter(FuncFormatter(_gbp))
    ax_static.set_xlim(0, x_top)
    ax_static.set_ylim(0, y_top)
    ax_static.grid(alpha=0.3)
    ax_static.legend(loc="lower right", fontsize=8, ncol=2, framealpha=0.85)
    ax_static.set_title(
        f"All 2^(n-1) = {2 ** n_mods} response curves for {focal_name}\n"
        "(active modifiers fixed at historical mean spend)"
    )

    x_silo, y_silo = curve_for(focal, (), {})
    x_sel, y_sel = curve_for(focal, subset, modifier_spends)

    dial_summary = ", ".join(
        f"{channel_names[j]}=\u00a3{modifier_spends[j]:,.0f}"
        for j in subset
    )
    ax_dyn.plot(
        x_silo, y_silo,
        color="#555", linewidth=2.5, linestyle="--",
        label="silo (no modifiers)", zorder=4,
    )
    ax_dyn.plot(
        x_sel, y_sel,
        color="#d62728", linewidth=2.8,
        label=f"{_subset_label(subset)} at {dial_summary}", zorder=5,
    )
    if show_silo:
        ax_dyn.scatter(
            hist_actual[:, focal], silo_hist[:, focal],
            color="#888", s=14, alpha=0.7, zorder=6,
            label="historical (silo)",
        )
    if show_obs:
        ax_dyn.scatter(
            hist_actual[:, focal], obs_hist[:, focal],
            color="black", s=22, alpha=0.85, zorder=7,
            label="historical (observed)",
        )
    ax_dyn.set_xlabel(f"{focal_name} weekly spend (\u00a3)")
    ax_dyn.set_ylabel(f"{focal_name} weekly contribution to sales (\u00a3)")
    ax_dyn.xaxis.set_major_formatter(FuncFormatter(_gbp))
    ax_dyn.yaxis.set_major_formatter(FuncFormatter(_gbp))
    ax_dyn.set_xlim(0, x_top)
    ax_dyn.set_ylim(0, y_top)
    ax_dyn.grid(alpha=0.3)
    ax_dyn.legend(loc="lower right", fontsize=9, framealpha=0.85)
    ax_dyn.set_title(
        "Silo vs. selected subset curve\n"
        "(pull every dial to \u00a30 to recover silo)"
    )

    fig.tight_layout()
    plt.show()


controls = {
    "focal_name": channel_dd,
    "subset": subset_dd,
    "show_silo": show_silo_cb,
    "show_obs": show_obs_cb,
}
controls.update({f"spend_{j}": modifier_sliders[j] for j in range(n_ch)})
out = widgets.interactive_output(_render, controls)

display(
    widgets.VBox(
        [
            widgets.HBox([channel_dd, subset_dd]),
            toggle_box,
            slider_box,
            out,
        ]
    )
)

## 14. Cross-channel elasticity

ROAS tells us how each channel performs in isolation. But once interactions are introduced, we can ask a richer question:

> If channel \(j\)'s spend increases by 1%, how much does **channel \(i\)'s contribution** change, holding everything else fixed?

This gives us a way to quantify the *interaction layer* in elasticity terms.

---

### Starting from the interaction model

Recall the observed contribution for channel \(i\):

$$
C_{i,t}
\;=\;
\beta_i\, m_{i,t}\,
\exp\!\Big(\sum_{j \ne i}\gamma_{ij}\,m_{j,t}\Big),
$$

where:

- \(m_{i,t}\) is the transformed media signal after max-scaling, adstock, and saturation,
- \(\beta_i\) is the baseline effectiveness of channel \(i\),
- \(\gamma_{ij}\) controls how channel \(j\) modifies the effectiveness of channel \(i\).

To isolate a single interaction pathway \(j \rightarrow i\), write:

$$
C_{i,t}
\;=\;
\beta_i\,m_{i,t}\,\exp(\gamma_{ij}m_{j,t}).
$$

Taking logs:

$$
\log C_{i,t}
=
\log(\beta_i m_{i,t})
+
\gamma_{ij}m_{j,t}.
$$

Holding all other terms fixed and differentiating with respect to transformed media \(m_{j,t}\):

$$
\frac{\partial \log C_{i,t}}{\partial m_{j,t}}
=
\gamma_{ij}.
$$

This tells us that \(\gamma_{ij}\) is the **semi-elasticity** of channel \(i\)'s contribution with respect to transformed media from channel \(j\).

---

## Why elasticity uses log-log derivatives

In econometrics, elasticity measures the proportional response of one variable to a proportional change in another:

$$
\text{Elasticity}
=
\frac{\%\ \text{change in output}}{\%\ \text{change in input}}.
$$

Using calculus:

$$
\text{Elasticity}
=
\frac{dC_i / C_i}{dx_j / x_j}.
$$

Logs provide a convenient mathematical representation of proportional changes because:

$$
d(\log x)
=
\frac{dx}{x}.
$$

That means:

$$
d\log x
$$

is approximately the infinitesimal percentage change in \(x\).

As a result, elasticity can be written compactly as:

$$
\boxed{
\frac{\partial \log C_i}{\partial \log x_j}
}
$$

which is the standard log-log elasticity form used throughout econometrics, demand modelling, and marketing response analysis.

This is one reason log-log specifications are so widely used in applied economics:

- coefficients become scale-free,
- effects become directly interpretable as percentage responses,
- nonlinear multiplicative relationships become linear in log space.

Classic applications include:

- Cobb-Douglas production functions,
- demand elasticity models,
- gravity models,
- advertising response functions,
- constant-elasticity marketing models.

---

## Moving from transformed media to actual spend

The model is not estimated directly on raw spend.

Instead, raw media spend is transformed through three stages:

$$
x_{j,t}
\;\rightarrow\;
\tilde x_{j,t}
\;\rightarrow\;
A_j(\tilde x_{j,t})
\;\rightarrow\;
S_j(A_j(\tilde x_{j,t}))
=
m_{j,t},
$$

where:

- \(x_{j,t}\) is raw spend,
- \(\tilde x_{j,t} = x_{j,t}/\max(x_j)\) is max-scaled spend,
- \(A_j(\cdot)\) is geometric adstock,
- \(S_j(\cdot)\) is Tanh saturation.

So transformed media is:

$$
m_{j,t}
=
S_j(A_j(\tilde x_{j,t})).
$$

This means the elasticity with respect to actual spend must account for:

1. max scaling,
2. adstock carryover,
3. nonlinear saturation.

Applying the chain rule:

$$
\frac{\partial \log C_{i,t}}{\partial \log x_{j,t}}
=
\frac{\partial \log C_{i,t}}{\partial m_{j,t}}
\cdot
\frac{\partial m_{j,t}}{\partial x_{j,t}}
\cdot
\frac{\partial x_{j,t}}{\partial \log x_{j,t}}.
$$

From above:

$$
\frac{\partial \log C_{i,t}}{\partial m_{j,t}}
=
\gamma_{ij}.
$$

Because:

$$
\frac{\partial x_{j,t}}{\partial \log x_{j,t}}
=
x_{j,t},
$$

we obtain:

$$
\frac{\partial \log C_{i,t}}{\partial \log x_{j,t}}
=
\gamma_{ij}
\cdot
x_{j,t}
\cdot
\frac{\partial m_{j,t}}{\partial x_{j,t}}.
$$

Substituting the transformed media definition:

$$
m_{j,t}
=
S_j(A_j(\tilde x_{j,t})),
$$

gives:

$$
\boxed{
\frac{\partial \log C_{i,t}}{\partial \log x_{j,t}}
=
\gamma_{ij}
\cdot
x_{j,t}
\cdot
\frac{\partial S_j(A_j(\tilde x_{j,t}))}{\partial x_{j,t}}
}
$$

This is the exact contemporaneous cross-channel elasticity with respect to raw spend.

---

## Interpreting the elasticity

The interaction elasticity now depends on:

- the interaction strength \(\gamma_{ij}\),
- the current spend level,
- the adstock state,
- and the local slope of the saturation curve.

This is an important result.

If channel \(j\) is already heavily saturated, then:

$$
\frac{\partial S_j(A_j(\tilde x_{j,t}))}{\partial x_{j,t}}
\approx 0,
$$

meaning a 1% increase in spend barely changes transformed media exposure, and therefore barely changes channel \(i\)'s effectiveness.

Conversely, when channel \(j\) is operating in the steep part of its response curve, interaction elasticities become much larger.

This produces a realistic property:

> interaction effects weaken naturally as channels saturate.

---

## Closed-form elasticity for Tanh saturation

For the Tanh saturation function:

$$
S_j(z)
=
b_j
\tanh\!\left(
\frac{z}{b_j c_j}
\right),
$$

the derivative is:

$$
S_j'(z)
=
\frac{1}{c_j}
\operatorname{sech}^2\!\left(
\frac{z}{b_j c_j}
\right).
$$

For contemporaneous spend effects under geometric adstock:

$$
\frac{\partial A_{j,t}}{\partial x_{j,t}}
=
\frac{1}{\max(x_j)},
$$

so the elasticity becomes:

$$
\boxed{
\frac{\partial \log C_{i,t}}{\partial \log x_{j,t}}
=
\gamma_{ij}
\cdot
\frac{x_{j,t}}{\max(x_j)}
\cdot
\frac{1}{c_j}
\operatorname{sech}^2\!\left(
\frac{A_{j,t}}{b_j c_j}
\right)
}
$$

Equivalently:

$$
\boxed{
\frac{\partial \log C_{i,t}}{\partial \log x_{j,t}}
=
\gamma_{ij}
\cdot
\tilde x_{j,t}
\cdot
S_j'(A_{j,t})
}
$$

where:

$$
\tilde x_{j,t}
=
\frac{x_{j,t}}{\max(x_j)}.
$$

---

## Worked examples

### Positive interaction

Suppose:

$$
\gamma_{12}=0.20,
\qquad
\tilde x_{2,t}=0.50,
\qquad
S_2'(A_{2,t})=0.40.
$$

Then:

$$
0.20 \times 0.50 \times 0.40
=
0.04.
$$

Interpretation:

> A 1% increase in channel_2 spend increases channel_1's contribution by approximately 0.04%.

---

### Negative interaction

Suppose:

$$
\gamma_{12}=-0.20,
\qquad
\tilde x_{2,t}=0.50,
\qquad
S_2'(A_{2,t})=0.40.
$$

Then:

$$
-0.20 \times 0.50 \times 0.40
=
-0.04.
$$

Interpretation:

> A 1% increase in channel_2 spend decreases channel_1's contribution by approximately 0.04%.

---

## Things to keep in mind

- This is the elasticity of **channel \(i\)'s contribution**, not total sales.

- It captures only the **indirect interaction effect** from channel \(j\) onto channel \(i\).

- The **total-sales elasticity** for channel \(j\) would also include:
  
  - channel \(j\)'s direct response,
  - its interaction effects on every other channel,
  - and any downstream propagation through the media ecosystem.

- These elasticities are **state-dependent** and **time-varying** because they depend on the current adstock and saturation state of the source channel.

- In practice, we therefore report the **average observed cross-channel elasticity** over the sample as a stable, comparable summary metric.

Elasticity measures:

> “If \(x\) changes by 1%, how much does \(y\) change in percentage terms?”

That is why elasticity uses logs.

The key identity is:

$$
d(\log x)
\approx
\frac{dx}{x},
$$

which means a small change in \(\log x\) is approximately a percentage change in \(x\).

So when we compute:

$$
\frac{\partial \log C_i}{\partial \log x_j},
$$

we are measuring:

> the percentage change in channel \(i\)'s contribution caused by a percentage change in channel \(j\)'s spend.

In our model (after chaining through max scaling and tanh saturation), we derived:

$$
\frac{\partial \log C_i}{\partial \log x_j}
=
\gamma_{ij}\,\tilde x_{j,t}\,S_j'(A_{j,t}).
$$

So \(\gamma_{ij}\,\tilde x_{j,t}\,S_j'(A_{j,t})\) is the cross-channel elasticity with respect to raw spend.

That means:

$$
\%\Delta C_i
\approx
\gamma_{ij}\,\tilde x_{j,t}\,S_j'(A_{j,t}) \times \%\Delta x_j.
$$

In [ ]:
# Cross-channel elasticity w.r.t. raw spend:
# η_ij,t = γ_ij · x̃_j,t · S'_j(A_j,t), with tanh saturation shortcut
# S'_j = (1/c_j)(1 - (m_j/b_j)^2).
# Posterior draws enter η before averaging over weeks.

gamma_post = idata.posterior["gamma"]
b_post = idata.posterior["saturation_b"]
c_post = idata.posterior["saturation_c"]
m_post = idata.posterior["transformed_media"]

gamma_arr = gamma_post.values  # (chain, draw, affected_i, source_j)
b_arr = b_post.values  # (chain, draw, channel)
c_arr = c_post.values
m_arr = m_post.values  # (chain, draw, date, channel)

mask = (1.0 - np.eye(n_channels))[np.newaxis, np.newaxis, np.newaxis, :, :]

s_prime_arr = (1.0 / c_arr[..., np.newaxis, :]) * (
    1.0 - (m_arr / b_arr[..., np.newaxis, :]) ** 2
)

tilde_x = truth.media_scaled.values  # (date, channel)

eta = (
    gamma_arr[..., np.newaxis, :, :]
    * mask
    * tilde_x[np.newaxis, np.newaxis, :, np.newaxis, :]
    * s_prime_arr[..., np.newaxis, :]
)

eta_post_mean = eta.mean(axis=(0, 1))
g_post_mean = gamma_arr.mean(axis=(0, 1)) * (1.0 - np.eye(n_channels))

avg_elast = eta_post_mean.mean(axis=0)
max_elast = np.sign(g_post_mean) * np.abs(eta_post_mean).max(axis=0)


def _direction_label(g: float) -> str:
    if g > 0:
        return "amplification"
    if g < 0:
        return "suppression"
    return "neutral"


rows = []
for i in range(n_channels):
    for j in range(n_channels):
        if i == j:
            continue
        g = float(g_post_mean[i, j])
        rows.append({
            "Source channel": truth.channel_names[j],
            "Affected channel": truth.channel_names[i],
            "Gamma mean": g,
            "Avg elasticity": float(avg_elast[i, j]),
            "Avg % change in affected contribution from 1% source increase": 100.0 * float(avg_elast[i, j]),
            "Max elasticity": float(max_elast[i, j]),
            "Max % change in affected contribution from 1% source increase": 100.0 * float(max_elast[i, j]),
            "Direction": _direction_label(g),
        })

elasticity_df = pd.DataFrame(rows).sort_values(
    "Avg % change in affected contribution from 1% source increase",
    key=lambda s: s.abs(),
    ascending=False,
).reset_index(drop=True)

elasticity_df.style.format({
    "Gamma mean": "{:+.4f}",
    "Avg elasticity": "{:+.4f}",
    "Avg % change in affected contribution from 1% source increase": "{:+.3f}%",
    "Max elasticity": "{:+.4f}",
    "Max % change in affected contribution from 1% source increase": "{:+.3f}%",
}).hide(axis="index")

> **How to read this table.** Read each row as: a 1% increase in the source
> channel is associated with an approximate X% change in the affected
> channel's contribution, through the interaction layer, at the average
> observed level of source-channel activity.

> **Caution.** These elasticities are local and conditional on the
> transformed media scale. They should be used to understand interaction
> strength, not as standalone ROAS or total sales elasticities.

## 15. Budget optimisation with interactions

In traditional MMM, budget optimisation reallocates spend across **independent** channel response curves: each channel's contribution depends only on its own spend.

In this interaction MMM, the optimiser must evaluate the **full media system** jointly, because each channel's **observed** scaled contribution depends on every other channel's transformed media through the multiplicative interaction layer.

Denote the scaled contribution (mean-scaled target scale) at week $t$ as

$$
\text{contribution}_t(x)
=
\alpha
+
\text{controls}_t
+
\text{seasonality}_t
+
\text{holidays}_t
+
\sum_i
\beta_i\, m_{i,t}(x)\,
\exp\!\Big(
\sum_{j \ne i}\gamma_{ij}\, m_{j,t}(x)
\Big)
$$

where $m_{i,t}(x)$ is the transformed media (max-scaling, geometric adstock with the same $l_{\max}$ as estimation, then tanh saturation) applied to the planned spend path encoded by $x$.

The optimisation objective over a future horizon $\mathcal{T}$ is **total expected scaled response** under a **posterior summary** (we plug in posterior means for all parameters below):

$$
\max_{\mathbf{b}}
\sum_{t \in \mathcal{T}}
\text{contribution}_t(\mathbf{b}),
\qquad
\mathbf{b} = (b_1, \dots, b_C)^\top.
$$

Where:

- $\mathbf{b}$ is the vector of **total budgets by channel** (GBP across the optimisation window $\mathcal{T}$);
- $\mathcal{T}$ is the optimisation horizon (weekly periods);
- intra-horizon timing for each channel is fixed by a **flighting matrix** $W$, not optimised.

We relate **planned GBP spend on week $t$** to $\mathbf{b}$ by

$$
x^{\text{future}}_{t,i} = b_i\, w_{t,i},
\qquad
\sum_{t \in \mathcal{T}} w_{t,i} = 1 \quad \forall i.
$$

Training max-scaling uses historical per-channel maxima $s_i = \max_\tau x^{\text{train}}_{\tau, i}$. The optimisation model maps planned GBP to the scaled input the model was trained on:

$$
\tilde{x}^{\text{future}}_{t,i}
=
\frac{x^{\text{future}}_{t,i}}{s_i}
=
\frac{b_i\, w_{t,i}}{s_i}.
$$

The flighting matrix $W$ has shape `(date, channel)` — here a `pandas.DataFrame` with `index = optimisation_dates`, `columns = channels`, non-negative entries, and **each column summing to one**. The optimiser chooses $\mathbf{b}$; $W$ is a planner knob that fixes **when** spend hits the funnel.

This keeps the optimisation **lightweight and planner-friendly**: we optimise budget shares across channels, not week-by-week channel allocation.


In [ ]:
# 15a. Optimiser helpers: wrapper + SLSQP optimiser classes.
#
#
# Crucially, _set_predictors_for_optimization here does NOT redefine the
# media stack: it clones the trained pm.Model and uses pm.set_data to
# swap in the optimisation-window inputs. The optimiser then composes
# pm.do({rv: posterior_mean, "media_scaled": planned}) on top so the
# graph that SLSQP sees IS the model's own `contribution` deterministic,
# just with free RVs frozen at their posterior means and the media data
# replaced by the budget-driven planned-spend expression.
from types import SimpleNamespace

from pymc.model.fgraph import clone_model
from pymc.model.transform.optimization import freeze_dims_and_data
from pytensor.graph import rewrite_graph
from scipy.optimize import minimize


class InteractionMMMWrapper:
    r"""Adapt the fitted interaction MMM to a BudgetOptimizer-style protocol."""

    def __init__(self, model, idata, truth, l_max=12):
        self.base_model = model
        self.idata = idata
        self.truth = truth
        self.l_max = l_max
        self.channel_columns = list(truth.channel_names)
        self.control_columns = list(truth.control_names)
        self.holiday_columns = list(truth.holiday_names)
        # Scaling factor used in training: s_i = max_t x^train_{t,i}.
        self._channel_scales = truth.media_actual.max(axis=0).values.astype(float)
        self.adstock = SimpleNamespace(l_max=l_max)

    def _set_predictors_for_optimization(
        self, num_periods, optimisation_dates, controls_scaled_at_dates=None,
    ):
        n_ch = len(self.channel_columns)

        if controls_scaled_at_dates is None:
            controls_window = np.tile(
                self.truth.controls_scaled.iloc[-1].to_numpy(dtype=float),
                (num_periods, 1),
            )
        else:
            controls_window = np.asarray(controls_scaled_at_dates, dtype=float)
            assert controls_window.shape == (num_periods, len(self.control_columns)), (
                f"controls_scaled_at_dates must be ({num_periods}, "
                f"{len(self.control_columns)}); got {controls_window.shape}"
            )

        dayofyear_window = optimisation_dates.dayofyear.to_numpy(dtype=float)
        holiday_distance_window = _nearest_event_distance(
            optimisation_dates, self.truth.holiday_dates,
        )

        m = clone_model(self.base_model)
        pm.set_data(
            {
                "media_scaled": np.zeros((num_periods, n_ch), dtype=float),
                "controls_scaled": controls_window,
                "holiday_distance": holiday_distance_window,
                "dayofyear": dayofyear_window,
            },
            model=m,
            coords={"date": optimisation_dates.to_numpy()},
        )
        return m


class InteractionBudgetOptimizer:
    r"""SLSQP budget optimiser for the interaction MMM.

    - Optimisation variable is the **channel-level total budget over the
      horizon** ($b_i$, GBP). Internally the solver works in *budget
      shares* (b_i / total_budget) for numerical stability.
    - Per-period spend is $b_i \cdot w_{t,i}$ where $w$ is a fixed
      ``budget_distribution_over_period`` (date by channel, columns
      summing to 1).
    - Solver: SciPy SLSQP with one equality constraint $\sum_i b_i = B$
      and box bounds $[\text{low}_i, \text{high}_i]$ per channel.

    The compiled objective walks the **trained model's own** ``contribution``
    deterministic after ``pm.do`` substitutes every free RV with its
    posterior mean and replaces ``media_scaled`` with the planned-spend
    PyTensor expression.
    """

    def __init__(
        self,
        wrapper,
        num_periods,
        total_budget,
        optimisation_dates,
        budget_distribution_over_period=None,
        bounds=None,
        controls_scaled_at_dates=None,
    ):
        self.wrapper = wrapper
        self.num_periods = num_periods
        self.total_budget = float(total_budget)
        self.optimisation_dates = optimisation_dates
        self.controls_scaled_at_dates = controls_scaled_at_dates

        if bounds is None:
            self.bounds = [(0.0, self.total_budget)] * len(wrapper.channel_columns)
        else:
            self.bounds = list(bounds)

        if budget_distribution_over_period is None:
            self.budget_distribution = pd.DataFrame(
                1.0 / num_periods,
                index=optimisation_dates,
                columns=wrapper.channel_columns,
            )
        else:
            assert np.allclose(budget_distribution_over_period.sum(axis=0), 1.0), (
                "budget_distribution_over_period columns must sum to 1"
            )
            self.budget_distribution = budget_distribution_over_period

        self._compiled = None
        self.result = None
        self.allocation = None
        self.allocation_strategy = None

    def _posterior_mean_substitutions(self, model):
        """Map every RV that contributes to ``contribution`` to a constant
        at its posterior mean."""
        post = self.wrapper.idata.posterior
        subs = {}
        for rv in model.free_RVs:
            name = rv.name
            if name in post.data_vars:
                subs[name] = pt.constant(post[name].mean(("chain", "draw")).values)
        return subs

    def _build_compiled_objective(self):
        if self._compiled is not None:
            return self._compiled

        wrapper = self.wrapper
        share_input = pt.dvector("share")
        B = pt.constant(self.total_budget)
        W_const = pt.constant(self.budget_distribution.to_numpy(dtype=float))
        scales_const = pt.constant(wrapper._channel_scales)
        planned_scaled = (share_input * B).dimshuffle("x", 0) * W_const / scales_const

        raw_model = wrapper._set_predictors_for_optimization(
            self.num_periods,
            self.optimisation_dates,
            controls_scaled_at_dates=self.controls_scaled_at_dates,
        )
        rv_subs = self._posterior_mean_substitutions(raw_model)

        frozen = freeze_dims_and_data(raw_model, data=[])
        graph = pm.do(frozen, {**rv_subs, "media_scaled": planned_scaled})

        objective = -rewrite_graph(
            pt.sum(graph["contribution"]),
            include=("lower_xtensor", "canonicalize", "stabilize"),
        )
        grad = pt.grad(objective, share_input)
        self._compiled = pm.compile_fn(
            [objective, grad], inputs=[share_input], model=graph,
        )
        return self._compiled

    def allocate_budget(self):
        compiled = self._build_compiled_objective()
        B = self.total_budget

        def _vg(share):
            v, g = compiled({"share": np.asarray(share, dtype=float)})
            return float(np.asarray(v)), np.asarray(g, dtype=float)

        n_ch = len(self.wrapper.channel_columns)
        share0 = np.full(n_ch, 1.0 / n_ch, dtype=float)
        share_bounds = [(lo / B, hi / B) for (lo, hi) in self.bounds]
        cons = (
            {
                "type": "eq",
                "fun": lambda share: float(np.sum(share) - 1.0),
                "jac": lambda share: np.ones_like(share, dtype=float),
            },
        )

        self.result = minimize(
            lambda share: _vg(share)[0],
            share0,
            jac=lambda share: _vg(share)[1],
            method="SLSQP",
            bounds=share_bounds,
            constraints=cons,
            options={"ftol": 1e-9, "maxiter": 1_000},
        )

        allocation_values = self.result.x * B
        self.allocation = pd.Series(
            allocation_values, index=self.wrapper.channel_columns, name="budget_GBP",
        )
        self.allocation_strategy = self.budget_distribution.multiply(
            self.allocation, axis=1,
        )
        return self.allocation, self.allocation_strategy


In [ ]:
# 15b. Optimise the **last 13 weeks of training data**, keeping the observed
# per-channel total spend and per-channel weekly flighting pattern as the
# baseline. The optimiser searches for a different *cross-channel* split of
# the same total spend, holding the per-channel weekly pattern (flighting)
# and all non-media drivers (controls, Fourier, holidays) fixed at their
# in-sample values over those 13 weeks.
hist_window = 13
hist_dates = truth.df.index[-hist_window:]
hist_media = truth.media_actual.iloc[-hist_window:]
hist_controls = truth.controls_scaled.iloc[-hist_window:]

baseline_budgets = hist_media.sum(axis=0).rename("baseline_actual_GBP")
total_budget = float(baseline_budgets.sum())

col_sums = hist_media.sum(axis=0).replace(0, np.nan)
flighting = hist_media.divide(col_sums, axis=1).fillna(1.0 / hist_window)
assert np.allclose(flighting.sum(axis=0), 1.0)

# Per-channel bounds at +/- 10% of the observed budget over the window.
bound_pct = 0.10
budget_bounds = [
    (float((1.0 - bound_pct) * b), float((1.0 + bound_pct) * b))
    for b in baseline_budgets.values
]

wrapper = InteractionMMMWrapper(model=model, idata=idata, truth=truth, l_max=12)
optimizer = InteractionBudgetOptimizer(
    wrapper=wrapper,
    num_periods=hist_window,
    total_budget=total_budget,
    optimisation_dates=hist_dates,
    budget_distribution_over_period=flighting,
    bounds=budget_bounds,
    controls_scaled_at_dates=hist_controls,
)

allocation, allocation_strategy = optimizer.allocate_budget()

compiled_obj = optimizer._build_compiled_objective()
baseline_share = (baseline_budgets / total_budget).to_numpy(dtype=float)
contribution_baseline_scaled = -float(np.asarray(compiled_obj({"share": baseline_share})[0]))
contribution_opt_scaled = -float(np.asarray(compiled_obj({"share": optimizer.result.x})[0]))

# Sanity: same-model check. The wrapper clones the trained pm.Model and
# only substitutes free RVs with their posterior means; nothing in the
# media stack (adstock, saturation, interactions, controls, Fourier,
# holidays, intercept) is redefined. So evaluating the trained model's
# `contribution` posterior-mean on the same window should agree with the
# wrapper's baseline up to adstock cold-start truncation.
trained_contrib_window_scaled = (
    idata.posterior["contribution"]
    .sel(date=hist_dates)
    .mean(("chain", "draw"))
    .sum()
    .item()
)

print({
    "window_weeks": hist_window,
    "window_dates": [str(hist_dates[0].date()), str(hist_dates[-1].date())],
    "total_budget_GBP": round(total_budget, 2),
    "baseline_total_scaled_contribution": round(contribution_baseline_scaled, 4),
    "optimised_total_scaled_contribution": round(contribution_opt_scaled, 4),
    "improvement_scaled": round(contribution_opt_scaled - contribution_baseline_scaled, 4),
    "trained_model_contribution_same_window": round(trained_contrib_window_scaled, 4),
    "n_iter": optimizer.result.nit,
})
print(
    "  ^ trained_model_contribution_same_window includes carry-over from "
    "all 91 pre-window training weeks; the wrapper's baseline is the same "
    "model but cold-started on the 13-week optimisation horizon (no media "
    "memory before week 1)."
)

n_ch = len(wrapper.channel_columns)
fig, ax = plt.subplots(figsize=(10, 4.2))
x_pos = np.arange(n_ch)
bar_w = 0.35
ax.bar(x_pos - bar_w / 2, baseline_budgets.values / 1e3, bar_w, label="baseline (actual)")
ax.bar(x_pos + bar_w / 2, allocation.values / 1e3, bar_w, label="optimised (SLSQP)")
ax.set_xticks(x_pos, wrapper.channel_columns)
ax.set_ylabel("GBP horizon budget (thousands)")
ax.set_title(f"Per-channel totals, last {hist_window} weeks")
ax.legend()
plt.tight_layout()
plt.show()

budget_df = pd.DataFrame({
    "baseline_actual_GBP": baseline_budgets,
    "optimised_GBP": allocation,
    "delta_GBP": allocation - baseline_budgets,
}).round(2)
budget_df


In [ ]:
# 15c. Optimiser validation: contribution improvement, bounds, budget equality.
share_opt = optimizer.result.x

contribution_opt_scaled = -float(np.asarray(compiled_obj({"share": share_opt})[0]))
contribution_baseline_scaled = -float(np.asarray(compiled_obj({"share": baseline_share})[0]))
delta_scaled = contribution_opt_scaled - contribution_baseline_scaled
pct_improvement = 100.0 * delta_scaled / contribution_baseline_scaled

target_mean_v = float(truth.target_mean)
contribution_opt_GBP = contribution_opt_scaled * target_mean_v
contribution_baseline_GBP = contribution_baseline_scaled * target_mean_v
delta_GBP = contribution_opt_GBP - contribution_baseline_GBP

print("--- 1. Total contribution improvement ----------------------------")
print(f"Baseline (actual) total scaled contribution: {contribution_baseline_scaled:>10.4f}")
print(f"Optimised total scaled contribution:         {contribution_opt_scaled:>10.4f}")
print(f"Improvement (scaled):                        {delta_scaled:>+10.4f}  ({pct_improvement:+.2f}%)")
print(f"Baseline (actual) contribution (GBP):        £{contribution_baseline_GBP:>16,.0f}")
print(f"Optimised contribution (GBP):                £{contribution_opt_GBP:>16,.0f}")
print(f"Improvement (GBP):                           £{delta_GBP:>+16,.0f}")

print()
print("--- 2. Bounds obeyed ---------------------------------------------")
opt_alloc = optimizer.allocation.to_numpy(dtype=float)
bounds_arr = np.asarray(optimizer.bounds, dtype=float)
bounds_tol = 1e-6
lower_slack = opt_alloc - bounds_arr[:, 0]
upper_slack = bounds_arr[:, 1] - opt_alloc
bounds_table = pd.DataFrame({
    "lower_GBP": bounds_arr[:, 0],
    "optimised_GBP": opt_alloc,
    "upper_GBP": bounds_arr[:, 1],
    "lower_slack": lower_slack,
    "upper_slack": upper_slack,
}, index=optimizer.wrapper.channel_columns).round(2)
print(bounds_table.to_string())
assert (lower_slack >= -bounds_tol).all(), f"Lower-bound violations: {lower_slack}"
assert (upper_slack >= -bounds_tol).all(), f"Upper-bound violations: {upper_slack}"
print(f"All per-channel bounds obeyed within {bounds_tol:.0e} GBP.")

print()
print("--- 3. Fixed total-budget equality obeyed ------------------------")
sum_opt = float(opt_alloc.sum())
budget_residual = sum_opt - optimizer.total_budget
budget_tol = 1e-3
print(f"Sum of optimised allocation: £{sum_opt:,.4f}")
print(f"Target total budget:         £{optimizer.total_budget:,.4f}")
print(f"Residual (sum - target):     £{budget_residual:+,.6f}")
assert abs(budget_residual) < budget_tol, (
    f"Budget equality constraint violated: residual={budget_residual}"
)
print(f"Total-budget equality obeyed within £{budget_tol:.0e}.")

# Sanity: the allocation strategy (date x channel) sums to the channel-level
# allocation over the date axis, since each flighting column sums to 1.
strategy_sum = optimizer.allocation_strategy.sum(axis=0).to_numpy(dtype=float)
assert np.allclose(strategy_sum, opt_alloc), (
    "allocation_strategy date-sum does not match allocation"
)
print("Flighting consistency: allocation_strategy.sum(date) == allocation.")


In [ ]:
# 15d. Future 13 weeks, equal flighting, fixed £1.3M total budget.
#
# Same wrapper / optimiser as the in-sample run, but applied to the 13
# weeks starting one week after the last training date. Flighting is
# uniform (1/13 per period per channel). Per-channel bounds reuse the
# in-sample baseline shape with a 20% bigger *upper* limit to accommodate
# the larger total budget (1.3M vs 1.26M in-window historical):
#   lower_i = 0.9  * baseline_i
#   upper_i = 1.1  * 1.20 * baseline_i = 1.32 * baseline_i
future_window = 13
future_dates = pd.date_range(
    truth.df.index.max() + pd.Timedelta(weeks=1),
    periods=future_window,
    freq="W-MON",
)
future_total_budget = 1_300_000.0

future_lower_pct = 0.10                                # match cell 38
future_upper_pct = (1.0 + future_lower_pct) * 1.20 - 1.0  # = +32% = +10% then 20% extra headroom
future_bounds = [
    (
        float((1.0 - future_lower_pct) * b),
        float((1.0 + future_upper_pct) * b),
    )
    for b in baseline_budgets.values
]

# Feasibility sanity: sum of lowers <= 1.3M <= sum of uppers.
sum_lower = sum(lo for lo, _ in future_bounds)
sum_upper = sum(hi for _, hi in future_bounds)
assert sum_lower <= future_total_budget <= sum_upper, (
    f"Bounds infeasible for total_budget={future_total_budget}: "
    f"sum_lower={sum_lower}, sum_upper={sum_upper}"
)

# Equal flighting: uniform 1/n_periods per channel.
future_flighting = pd.DataFrame(
    1.0 / future_window,
    index=future_dates,
    columns=wrapper.channel_columns,
)

future_optimizer = InteractionBudgetOptimizer(
    wrapper=wrapper,
    num_periods=future_window,
    total_budget=future_total_budget,
    optimisation_dates=future_dates,
    budget_distribution_over_period=future_flighting,
    bounds=future_bounds,
)

future_allocation, future_strategy = future_optimizer.allocate_budget()
future_compiled = future_optimizer._build_compiled_objective()

# Baseline for comparison: pro-rata scaling of the in-window historical
# split to the new total. (An equal £325k/channel split is infeasible
# under these bounds — e.g. channel_2 has lower bound £509k — so a
# pro-rata baseline is the planner-meaningful counterpart.)
prorata_alloc = (baseline_budgets / baseline_budgets.sum()) * future_total_budget
prorata_share = (prorata_alloc / future_total_budget).to_numpy(dtype=float)
share_opt = future_optimizer.result.x

contribution_prorata_scaled = -float(np.asarray(future_compiled({"share": prorata_share})[0]))
contribution_opt_scaled = -float(np.asarray(future_compiled({"share": share_opt})[0]))
delta_scaled = contribution_opt_scaled - contribution_prorata_scaled
pct_improvement = 100.0 * delta_scaled / contribution_prorata_scaled
target_mean_v = float(truth.target_mean)

print({
    "future_window_weeks": future_window,
    "future_dates": [str(future_dates[0].date()), str(future_dates[-1].date())],
    "future_total_budget_GBP": future_total_budget,
    "bounds_pct_of_in_sample_baseline": [-future_lower_pct, future_upper_pct],
    "pro_rata_total_scaled_contribution": round(contribution_prorata_scaled, 4),
    "optimised_total_scaled_contribution": round(contribution_opt_scaled, 4),
    "improvement_scaled_vs_prorata": round(delta_scaled, 4),
    "improvement_pct": round(pct_improvement, 2),
    "improvement_GBP": round(delta_scaled * target_mean_v, 0),
    "n_iter": future_optimizer.result.nit,
})

# Inline validation: bounds + budget equality.
opt_alloc_arr = future_optimizer.allocation.to_numpy(dtype=float)
bounds_arr = np.asarray(future_optimizer.bounds, dtype=float)
assert (opt_alloc_arr >= bounds_arr[:, 0] - 1e-6).all(), "lower-bound violation"
assert (opt_alloc_arr <= bounds_arr[:, 1] + 1e-6).all(), "upper-bound violation"
assert abs(opt_alloc_arr.sum() - future_optimizer.total_budget) < 1e-3, "budget equality violation"
strategy_sum = future_strategy.sum(axis=0).to_numpy(dtype=float)
assert np.allclose(strategy_sum, opt_alloc_arr), "flighting consistency violation"
print("Validation: bounds, budget equality, and flighting consistency all hold.")

# Plots.
n_ch = len(wrapper.channel_columns)
fig, ax = plt.subplots(figsize=(10, 4.2))
x_pos = np.arange(n_ch)
bar_w = 0.35
ax.bar(x_pos - bar_w / 2, prorata_alloc.values / 1e3, bar_w, label="pro-rata of last 13 wks")
ax.bar(x_pos + bar_w / 2, future_allocation.values / 1e3, bar_w, label="optimised (SLSQP)")
for c, (lo, hi) in enumerate(future_bounds):
    ax.hlines(lo / 1e3, c - 0.45, c + 0.45, colors="grey", linestyles=":", linewidth=1)
    ax.hlines(hi / 1e3, c - 0.45, c + 0.45, colors="grey", linestyles=":", linewidth=1)
ax.set_xticks(x_pos, wrapper.channel_columns)
ax.set_ylabel("GBP horizon budget (thousands)")
ax.set_title(f"Future {future_window}-week budget split (£{future_total_budget/1e6:.1f}M total)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

future_budget_df = pd.DataFrame({
    "lower_bound_GBP": [b[0] for b in future_bounds],
    "pro_rata_GBP": prorata_alloc,
    "optimised_GBP": future_allocation,
    "upper_bound_GBP": [b[1] for b in future_bounds],
    "delta_vs_prorata_GBP": future_allocation - prorata_alloc,
}, index=wrapper.channel_columns).round(2)
future_budget_df
